## Planning:
- Load data
- Check variable and target
- Check correlation between numeric variable

In [2]:
# data processing
import pandas as pd
import numpy as np

# data visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# data modeling 
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# model 
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submit = pd.read_csv("sample_submission.csv")

In [4]:
train['Arrival Delay in Minutes'] = train['Arrival Delay in Minutes'].fillna(train['Departure Delay in Minutes'])

In [5]:
test['Arrival Delay in Minutes'] = test['Arrival Delay in Minutes'].fillna(test['Departure Delay in Minutes'])

In [6]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()

In [7]:
non_numeric_cols = train.select_dtypes(exclude='number').columns.tolist()

In [8]:
oof_logit_1 = pd.read_csv("blending\oof_logit_1_5cv.csv")
oof_xgb_2 = pd.read_csv("blending\oof_xgb_2_5cv.csv")
oof_xgb_6 = pd.read_csv("blending\oof_xgb_6_5cv_optuna.csv")
oof_xgb_7 = pd.read_csv("blending\oof_xgb_7_20cv_optuna.csv")
oof_xgb_10 = pd.read_csv("blending\oof_xgb_10_5cv_fe.csv")

In [9]:
sub_logit_1 = pd.read_csv("blending\sub_logit_1.csv")
sub_xgb_2 = pd.read_csv("blending\sub_xgb_2.csv")
sub_xgb_6 = pd.read_csv("blending\sub_xgb_6_5cv_optuna.csv")
sub_xgb_7 = pd.read_csv("blending\sub_xgb_7_20cv_optuna.csv")
sub_xgb_10 = pd.read_csv("blending\sub_xgb_10_5cv_feature_engineering.csv")

In [10]:
sub_logit_1.rename(columns={"satisfaction": oof_logit_1.columns.to_list()[2]}, inplace=True)
sub_xgb_2.rename(columns={"satisfaction": oof_xgb_2.columns.to_list()[2]}, inplace=True)
sub_xgb_6.rename(columns={"satisfaction": oof_xgb_6.columns.to_list()[2]}, inplace=True)
sub_xgb_7.rename(columns={"satisfaction": oof_xgb_7.columns.to_list()[2]}, inplace=True)
sub_xgb_10.rename(columns={"satisfaction": oof_xgb_10.columns.to_list()[2]}, inplace=True)

## Basic Data Process for Modeling

Data preparation - Create Dummy variable

In [11]:
# train_dummy = pd.get_dummies(train, columns=non_numeric_cols, dtype=int)
train_dummy = pd.get_dummies(train, columns=non_numeric_cols,drop_first=True, dtype=int)

In [12]:
test_dummy = pd.get_dummies(test, columns=non_numeric_cols.remove('satisfaction'),drop_first=True, dtype=int)

In [13]:
'''
X = train_dummy.drop(columns=["id","satisfaction_True"])
y = train_dummy["satisfaction_True"]
X_test = test_dummy.drop(columns=["id"])
'''

'\nX = train_dummy.drop(columns=["id","satisfaction_True"])\ny = train_dummy["satisfaction_True"]\nX_test = test_dummy.drop(columns=["id"])\n'

In [14]:
train_data = train_dummy[["id","satisfaction_True"]]

In [15]:
train_data = pd.concat([train_data,oof_logit_1.iloc[:,[2]]],axis=1)

In [16]:
train_data = pd.concat([train_data,oof_xgb_2.iloc[:,[2]],oof_xgb_6.iloc[:,[2]],oof_xgb_7.iloc[:,[2]],oof_xgb_10.iloc[:,[2]]],axis=1)

In [17]:
train_data.head(2)

,id,satisfaction_True,oof_logit_1_5cv,oof_xgb_2_5cv,oof_xgb_6_5cv_optuna,oof_xgb_7_20cv_optuna,oof_xgb_10_5cv_fe
0,0,0,0.089254,0.264970,0.257997,0.232633,0.246623
1,1,1,0.931650,0.957369,0.957024,0.950261,0.973524


In [18]:
test_data = test_dummy[["id"]]

In [19]:
test_data = pd.concat([test_data,sub_logit_1.iloc[:,[1]],sub_xgb_2.iloc[:,[1]],sub_xgb_6.iloc[:,[1]],sub_xgb_7.iloc[:,[1]],sub_xgb_10.iloc[:,[1]]],axis=1)

In [20]:
test_data.head(2)

,id,oof_logit_1_5cv,oof_xgb_2_5cv,oof_xgb_6_5cv_optuna,oof_xgb_7_20cv_optuna,oof_xgb_10_5cv_fe
0,699635,0.089525,0.236491,0.230871,0.229289,0.227331
1,699636,0.582690,0.899090,0.890153,0.891726,0.860838


In [21]:
X = train_data.drop(columns=["id","satisfaction_True"])
y = train_data["satisfaction_True"]
X_test = test_data.drop(columns=["id"])

## Find Optimum Parameter using Optuna for XGBOOST

In [1]:
import optuna

In [22]:
SEED = 42

In [23]:
# define objective using XGBoost
# Step 1: Define the Optuna objective with TimeSeriesSplit on x_train
def objective(trial):
    np.random.seed(42)
    params = {
        # fix parameter
        "objective":"binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 10000,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": SEED,
        "colsample_bytree":0.8,
        "subsample":0.8,
        "n_jobs":-1,

        # trial parameter
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2,log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0,log=True),
        "reg_alpha": trial.suggest_float("alpha", 1e-8, 1.0,log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 30),

        # early stopping round
        "early_stopping_rounds": 200,
    }
    
    train_idx = int(0.85 * X.shape[0])

    x_tr, x_val = X.iloc[:train_idx], X.iloc[train_idx:]
    y_tr, y_val = y.iloc[:train_idx], y.iloc[train_idx:]

    model = xgb.XGBClassifier(**params)
    model.fit(x_tr, y_tr,
              eval_set=[(x_val, y_val)],
              verbose =False)

    preds = model.predict_proba(x_val)[:,1]
    # rmse = np.sqrt(root_mean_squared_error(y_val, preds))
    score_auc = roc_auc_score(y_val, preds)

    # Save best iteration
    trial.set_user_attr("best_iteration", model.best_iteration)

    return score_auc

In [24]:
# Step 2: Run the Optuna study
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=150)

[I 2026-10-30 20:16:26,209] A new study created in memory with name: no-name-70530abb-1863-41c0-aba6-b5a0a5edbaf2
[I 2026-10-30 20:16:43,846] Trial 0 finished with value: 0.9605787381187074 and parameters: {'learning_rate': 0.030710573677773714, 'max_depth': 8, 'reg_lambda': 0.007177141927992002, 'alpha': 0.0006155564318973012, 'min_child_weight': 9}. Best is trial 0 with value: 0.9605787381187074.
[I 2026-10-30 20:17:40,573] Trial 1 finished with value: 0.9606529938508499 and parameters: {'learning_rate': 0.015957084694148364, 'max_depth': 3, 'reg_lambda': 0.08499808989182997, 'alpha': 0.0006440507553993703, 'min_child_weight': 23}. Best is trial 1 with value: 0.9606529938508499.
[I 2026-10-30 20:18:06,510] Trial 2 finished with value: 0.9605937658541854 and parameters: {'learning_rate': 0.010636066512540286, 'max_depth': 8, 'reg_lambda': 0.04566054873446119, 'alpha': 4.997040685255803e-07, 'min_child_weight': 9}. Best is trial 1 with value: 0.9606529938508499.
[I 2026-10-30 20:18:59,

In [25]:
# Step 4: Train final model on full x_train with best parameters
best_params = study.best_params
best_trial = study.best_trial

print("Best Value (AUC):", study.best_value)
print("Best Params:", best_params)
print("Best iteration (trees):", best_trial.user_attrs["best_iteration"])

Best Value (AUC): 0.9607078523910817
Best Params: {'learning_rate': 0.05892906997827939, 'max_depth': 4, 'reg_lambda': 1.1966459664586488e-05, 'alpha': 1.0510871416902545e-08, 'min_child_weight': 9}
Best iteration (trees): 243


In [26]:
final_params = {
    # fix parameter
        "objective":"binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 10000,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": SEED,
        "colsample_bytree":0.8,
        "subsample":0.8,
        "n_jobs":-1,
        # early stopping round
        "early_stopping_rounds": 200,

        **best_params
}

## Model using XGboost 5 fold CV

In [27]:
SEED = 42
N_FOLDS = 5
TARGET = 'satisfaction_True'

In [28]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [29]:
def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [30]:
def fit_xgb(X_tr, y_tr, X_va, y_va):
    model = xgb.XGBClassifier(**final_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [31]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

  fold 0: AUC = 0.96049
  fold 1: AUC = 0.95931
  fold 2: AUC = 0.96107
  fold 3: AUC = 0.96041
  fold 4: AUC = 0.96037
XGBoost | OOF AUC = 0.95966 (mean 0.96033 ± 0.00057)


In [32]:
def save_sub(pred, name):
    sub = submit.copy()
    sub['satisfaction'] = pred
    sub.to_csv(name, index=False)
    print('saving file:', name)

In [36]:
len(test_xgb)

299844

In [33]:
save_sub(test_xgb,'sub_xgb_12_5cv_stacking_optuna.csv')

saving file: sub_xgb_12_5cv_stacking_optuna.csv


In [37]:
oof_xgb_df = train[['id']].copy()
oof_xgb_df.loc[:,'oof_xgb_12_20cv_stacking_optuna'] = oof_xgb
oof_xgb_df.to_csv('oof_xgb_12_20cv_stacking_optuna.csv')

In [82]:
auc_xgb

0.945144978672748